In [3]:
import sqlite3

In [109]:
class Catalog:
    def __init__(self, dbfile):
        print("Creating Catalog")
        self.conn = sqlite3.connect(dbfile)
        self.cursor = self.conn.cursor()
        self.setup()

        
    def setup(self):

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY,
            brand TEXT NOT NULL,
            name TEXT NOT NULL,
            variant TEXT NOT NULL,
            UNIQUE(brand, name, variant)
        );
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS prices (
        id INTEGER PRIMARY KEY,
        product_id INTEGER NOT NULL,
        price REAL NOT NULL,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,

        FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE CASCADE
        );
        """)
        
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS product_urls (
        id INTEGER PRIMARY KEY,
        product_id INTEGER NOT NULL,
        url TEXT NOT NULL,
        
        FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE CASCADE
        );
        """)


        self.conn.commit()

        
    def addProduct(self, brand:str, name: str, variant:str, urls:list):
        try:
            self.cursor.execute(
                """
                INSERT INTO products (brand, name, variant)
                VALUES (?, ?, ?)
                """,
                (brand, name, variant),
            )

            product_id = self.cursor.lastrowid

            for url in urls:
                self.cursor.execute(
                    """
                    INSERT INTO product_urls (product_id, url)
                    VALUES (?, ?)
                    """,
                    (product_id, url),
                )

            self.conn.commit()

        except Exception:
            self.conn.rollback()
            # raise

    
    def getAllBrands(self):
        
        self.cursor.execute("""
            SELECT DISTINCT brand
            FROM products
            ORDER BY brand
        """)
        rows = self.cursor.fetchall()
        # Flatten to a simple list of strings
        return [row[0] for row in rows]


    
    def getProductsByBrand(self, brand: str):
    
        self.cursor.execute("""
            SELECT p.id, p.name, p.variant, u.url
            FROM products p
            JOIN product_urls u
                ON p.id = u.product_id
            WHERE p.brand = ?
            ORDER BY p.name
        """, (brand,))
    
        rows = self.cursor.fetchall()
        products = {}

        for product_id, name, variant, url in rows:
            if product_id not in products:
                    products[product_id] = {
                        "name": name,
                        "variant": variant,
                        "urls": []
                    }
    
            products[product_id]["urls"].append(url)

        products = {k: v for k, v in sorted(products.items(), key=lambda item: item[0])}
        return products

    
        # return products

In [110]:
cat = Catalog('catalog.db')

Creating Catalog


In [111]:
cat.getAllBrands()

['Lelo']

In [149]:
cat.addProduct("Lelo", "Liv 3", "Deep Rose", ["https://www.lelo.com/liv-3", 
                                              "https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YS4497"])
cat.addProduct("Lelo", "Liv 3", "Powder Blue", ["https://www.lelo.com/liv-3", 
                                              "https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YQNLL8"])
cat.addProduct("Lelo", "Gigi 3", "Powder Blue", ["https://www.lelo.com/gigi-3"])

cat.addProduct("Bellesa", "Aurora 2", "", ["https://www.bboutique.co/sex-toys/vibrators/aurora-2-8158232412405"])

In [180]:
cat.getProductsByBrand("Lelo")

{1: {'name': 'Liv 3',
  'variant': 'Deep Rose',
  'urls': ['https://www.lelo.com/liv-3',
   'https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YS4497']},
 2: {'name': 'Liv 3',
  'variant': 'Powder Blue',
  'urls': ['https://www.lelo.com/liv-3',
   'https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YQNLL8']},
 3: {'name': 'Gigi 3',
  'variant': 'Powder Blue',
  'urls': ['https://www.lelo.com/gigi-3']}}

In [181]:
cat.getProductsByBrand("Bellesa")

{4: {'name': 'Aurora 2',
  'variant': '',
  'urls': ['https://www.bboutique.co/sex-toys/vibrators/aurora-2-8158232412405']}}

In [152]:
UrlToPrice = {}


In [163]:
from urllib.parse import urlparse

def get_domain_from_url(url):
    # Add a scheme if missing, so urlparse can correctly identify the netloc
    if '://' not in url:
        url = 'http://' + url
        
    parsed_url = urlparse(url)
    # The .netloc attribute contains the domain and potentially a port number
    domain = parsed_url.netloc
    
    # Remove port number if present (e.g., 'example.com:8080')
    if ':' in domain:
        domain = domain.split(':')[0]
    if domain.startswith("www."):
        domain = domain[4:]
    return domain

def normalizePrice(text):
    if "." not in text:
        text = text+".00"
    if text.startswith("$"):
        text = text.replace("$", "USD ")
    return text

In [167]:
# from playwright.sync_api import sync_playwright
from playwright.async_api import async_playwright


async def parse_amazon(page):
    cssClasses = ["a-offscreen"]
    cls_selector = ', '.join(f"[class*='{c}']" for c in cssClasses) 
    await page.wait_for_selector(cls_selector)
    text = await page.locator(cls_selector).first.text_content()
    
    return normalizePrice(text)


async def parse_lelo(page):
    cssClasses = ["EmphasizedPrice", "RegularPrice"]
    cls_selector = ', '.join(f"[class*='{c}']" for c in cssClasses) 
    await page.wait_for_selector(cls_selector)
    text = await page.locator(cls_selector).first.text_content()
    
    return normalizePrice(text)



async def parse_bboutique(page):
    cssClasses = ["kqvwUh"]
    cls_selector = ', '.join(f"[class*='{c}']" for c in cssClasses) 
    await page.wait_for_selector(cls_selector)
    text = await page.locator(cls_selector).first.text_content()
    return normalizePrice(text)




PARSER_MAP = {
    "amazon.com": parse_amazon,
    "lelo.com": parse_lelo,
    "bboutique.co": parse_bboutique
}


async def get_price(url):
    domain = get_domain_from_url(url)  
    parser = PARSER_MAP.get(domain)
    if parser is None:
        print("No parser for domain:", domain)
        return

        
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(url)      
        try:
            price = await parser(page)
            print(price, f"[{url}]")
            UrlToPrice[url] = price
        except Exception as e:
            print(f"Error parsing {url}: {e}")

        await browser.close()

# await get_price("https://www.lelo.com/gigi-3")

In [176]:
def getUniqueUrls():
    uniqueUrls = []
    for brand in cat.getAllBrands():
        for key, val in cat.getProductsByBrand(brand).items():
            uniqueUrls.extend(val["urls"])
    return uniqueUrls

In [177]:
getUniqueUrls()

['https://www.bboutique.co/sex-toys/vibrators/aurora-2-8158232412405',
 'https://www.lelo.com/liv-3',
 'https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YS4497',
 'https://www.lelo.com/liv-3',
 'https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YQNLL8',
 'https://www.lelo.com/gigi-3']

In [179]:
for u in getUniqueUrls():
    await get_price(u)
    

USD 59.00 [https://www.bboutique.co/sex-toys/vibrators/aurora-2-8158232412405]
USD 109.00 [https://www.lelo.com/liv-3]
USD 89.38 [https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YS4497]
USD 109.00 [https://www.lelo.com/liv-3]
USD 109.00 [https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YQNLL8]
USD 116.07 [https://www.lelo.com/gigi-3]
